Updated inference defaults: local labels use empirical adjusted p-values; zero permutations are untested. ICG uses within-type contact flags and an explicit expression scale. Recorded outputs were cleared and require rerunning. See `docs/analysis.md` for assumptions.


# `tl.interaction_changed_genes`: vectorization-equivalence check vs a per-gene `scipy.stats` loop

**Scope, stated honestly:** this notebook is a *vectorization-equivalence check*, not an external validation. The "reference" below re-types astrolabe's own algorithm — the same contacted/non-contacted split, the same zero-variance guard, the same test choice — as a plain per-gene `scipy.stats.mannwhitneyu` loop. What it validates is that astrolabe's all-genes-at-once vectorized Mann-Whitney implementation (ranking, tie correction, continuity correction, normal approximation) reproduces scipy's per-gene arithmetic exactly; it cannot detect a flaw in the *method design*, because both sides share it by construction. The real external reference for this analysis is Giotto's `findInteractionChangedGenes` (Dries et al. 2021, Genome Biology) — an R package, out of scope for this Python benchmark suite.

For every (receiver cell type, neighbor cell type) pair, `interaction_changed_genes` tests whether "contacted" cells (>=1 spatial neighbor of the given type) differ in expression from non-contacted cells of the same type. Since the overhaul, astrolabe's default **`pval` is a within-type contact-flag permutation p-value** (cells in tissue are not exchangeable, so the asymptotic p is anti-conservative under spatial autocorrelation — exchangeability within the specified strata is required; this is not an exact Giotto implementation); the asymptotic Mann-Whitney p is still reported as **`p_asymptotic`**, and that is the scipy-comparable column this notebook checks. The permutation path itself has no independent Python implementation to compare against and is exercised by unit tests, not validated here.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

import time
import numpy as np
import pandas as pd
import astrolabe as ab
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

## Dataset

`ab.datasets.xenium_tiny()` with Leiden clusters as cell types and a spatial
graph to define "contact."

In [ ]:
mdata = ab.datasets.xenium_tiny()
ab.pipeline.filter_xenium(mdata)
ab.pp.normalize_total(mdata)
ab.pp.log1p(mdata)
ab.pp.highly_variable_genes(mdata)
ab.tl.pca(mdata)
ab.tl.transcriptional_neighbors(mdata)
ab.tl.cluster(mdata, resolution=0.5)
ab.tl.spatial_neighbors(mdata)

adata = mdata["xenium"]
adata

## Astrolabe: `tl.interaction_changed_genes`

Run with the default permutation null (`n_perms=200`), so the returned frame carries both the default `pval` (permutation) and `p_asymptotic` (the vectorized Mann-Whitney normal approximation under comparison here).

In [ ]:
_t0 = time.perf_counter()
icg_astro = ab.tl.interaction_changed_genes(
    mdata, expression_scale="log1p", groupby="leiden", method="wilcoxon", fdr_scope="pair",
)
t_astro_s = time.perf_counter() - _t0
print(f"astrolabe runtime (incl. 200-permutation null): {t_astro_s:.4f}s")
icg_astro.head()

## Reference: the identical contacted/non-contacted split, tested with plain `scipy.stats`

Reproduces the same contact split astrolabe computes (any nonzero connectivity weight from a receiver cell to a neighbor-type cell counts as "contacted"; self pairs skipped, matching astrolabe's `skip_self_pairs=True` default), then runs `scipy.stats.mannwhitneyu` per gene with `method='asymptotic'` and continuity correction — again matching astrolabe's implementation by construction (that's what makes this an equivalence check, not a validation).

One edge case needs explicit handling: for a gene that's constant (e.g. all zero) across both groups combined, the Mann-Whitney normal approximation has zero variance and plain `scipy.stats.mannwhitneyu` returns `nan`. Astrolabe guards this case and reports `p=1.0` instead, so the reference loop applies the same guard to match.

In [ ]:
categories = adata.obs["leiden"].cat.categories.tolist()
conn = adata.obsp["spatial_neighbors_connectivities"]
X = np.asarray(adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X, dtype=np.float64)
gene_names = list(adata.var_names)
codes = adata.obs["leiden"].cat.codes.to_numpy()
type_idx = {ct: np.where(codes == i)[0] for i, ct in enumerate(categories)}

_t0 = time.perf_counter()
rows = []
for ct_a in categories:
    idx_a = type_idx[ct_a]
    conn_a = conn[idx_a, :]
    for ct_b in categories:
        if ct_a == ct_b:
            continue  # matches astrolabe's skip_self_pairs=True default
        idx_b = type_idx[ct_b]
        if len(idx_b) == 0:
            continue
        neighbor_counts = np.asarray(conn_a[:, idx_b].sum(axis=1)).ravel()
        local_cont = np.where(neighbor_counts > 0)[0]
        local_non = np.where(neighbor_counts == 0)[0]
        if len(local_cont) < 10 or len(local_non) < 10:
            continue

        X_cont = X[idx_a[local_cont], :]
        X_non = X[idx_a[local_non], :]

        pvals = np.empty(len(gene_names))
        for g in range(len(gene_names)):
            col_cont, col_non = X_cont[:, g], X_non[:, g]
            if np.ptp(np.concatenate([col_cont, col_non])) == 0:
                # Zero-variance combined column: matches astrolabe's own
                # sigma > 0 guard rather than propagating scipy's nan.
                pvals[g] = 1.0
            else:
                _, p = mannwhitneyu(
                    col_cont, col_non,
                    method="asymptotic", use_continuity=True,
                )
                pvals[g] = p
        pvals_adj = multipletests(pvals, method="fdr_bh")[1]

        for g, gene in enumerate(gene_names):
            rows.append({
                "cell_type": ct_a, "neighbor_type": ct_b, "gene": gene,
                "pval_ref": pvals[g], "pval_adj_ref": pvals_adj[g],
            })

t_ref_s = time.perf_counter() - _t0
print(f"reference runtime: {t_ref_s:.4f}s")

icg_ref = pd.DataFrame(rows)
icg_ref.head()

## Agreement (on `p_asymptotic` — the scipy-comparable column)

`pval` and `pval_adj` are permutation-based since the overhaul and have no scipy counterpart, so the comparison targets `p_asymptotic`. The reference's BH-adjusted values are checked against a BH re-application to astrolabe's own `p_asymptotic` (statsmodels vs astrolabe's internal BH would otherwise be conflated with the p-value comparison).

In [ ]:
merged = icg_astro.merge(icg_ref, on=["cell_type", "neighbor_type", "gene"])
assert len(merged) == len(icg_astro), "pair sets should match exactly"

# pvals can be exactly 0 in floating point (deep in the tail of the normal
# approximation); clip before -log10 so the comparison isn't dominated by a
# handful of -inf/+inf outliers that both sides agree are "very significant."
eps = 1e-300
neglog_astro = -np.log10(np.clip(merged["p_asymptotic"], eps, 1.0))
neglog_ref = -np.log10(np.clip(merged["pval_ref"], eps, 1.0))

r = np.corrcoef(neglog_astro, neglog_ref)[0, 1]
max_diff = float((merged["p_asymptotic"] - merged["pval_ref"]).abs().max())

# BH re-applied per pair to astrolabe's asymptotic p, vs statsmodels' BH
bh_astro = merged.groupby(["cell_type", "neighbor_type"], observed=True)[
    "p_asymptotic"].transform(lambda p: multipletests(p, method="fdr_bh")[1])
adj_max_diff = float((bh_astro - merged["pval_adj_ref"]).abs().max())

print(f"Pearson r (-log10 p_asymptotic) = {r:.6f}")
print(f"max |p_asymptotic - reference p| = {max_diff:.2e}")
print(f"max |BH(p_asymptotic) - reference BH| = {adj_max_diff:.2e}")
print(f"Runtime: astrolabe {t_astro_s:.4f}s (incl. permutation null) vs "
      f"per-gene scipy loop {t_ref_s:.4f}s (asymptotic only) — not "
      f"apples-to-apples: astrolabe's call also ran a 200-permutation null "
      f"the reference skips.")

assert max_diff < 1e-10, "vectorized Mann-Whitney must reproduce scipy exactly"

panel = ab.pl.scatter(
    adata, x=neglog_astro.to_numpy(), y=neglog_ref.to_numpy(), annotate_corr=True,
    xlabel="astrolabe p_asymptotic (-log10)",
    ylabel="scipy.stats.mannwhitneyu (-log10 p)",
    title="ICG asymptotic-p equivalence, all (pair, gene) rows",
)
panel.show()

## Interpretation

Both sides run the identical asymptotic, tie-corrected, continuity-corrected Mann-Whitney U test on the identical contacted/non-contacted split, so `p_asymptotic` lands on `y = x` to floating-point precision — confirming the vectorized ranking/tie-correction arithmetic, which is all this notebook can confirm. It says nothing about whether the *method* (the contact split, or the choice of test) is sound, because the reference re-types that method rather than implementing it independently; and it does not touch astrolabe's default significance path (`pval`, the within-type contact-flag permutation null), which exists precisely because the asymptotic p compared here is anti-conservative under spatial autocorrelation. An external validation would compare against Giotto's `findInteractionChangedGenes` (R) on a shared dataset — out of scope for this Python-only benchmark suite.